In [3]:
import xml.etree.ElementTree as ET
import pandas as pd
import glob
import os
import csv


In [4]:
def get_txt(node, path):
    """Función auxiliar para extraer texto de forma segura."""
    if node is None: return None
    target = node.find(path)
    return target.text if target is not None else None


In [22]:
# ==========================================
# FUNCIÓN DE EXTRACCIÓN (STREAMING)
# ==========================================
def ejecutar_extraccion():
    print(f"🚀 Iniciando extracción de {PROVINCIA}...")
    print(f"📂 Origen: {XML_ENTRADA}")
    print(f"📄 Destino: {CSV_SALIDA}")

    if not os.path.exists(XML_ENTRADA):
        print(f"❌ Error: No se encuentra el archivo {XML_ENTRADA}")
        return

    with open(CSV_SALIDA, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNAS)
        writer.writeheader()

        # iterparse con limpieza de memoria
        context = ET.iterparse(XML_ENTRADA, events=('end',))
        count = 0

        for event, elem in context:
            if elem.tag == 'DatosEnergeticosDelEdificio':
                # --- Bloques principales ---
                ident = elem.find('IdentificacionEdificio')
                geom = elem.find('DatosGeneralesyGeometria')
                inst = elem.find('InstalacionesTermicas')
                renov = elem.find('EnergiasRenovables')
                demanda = elem.find('Demanda/EdificioObjeto')
                consumo = elem.find('Consumo/EnergiaPrimariaNoRenovable')
                emisiones = elem.find('EmisionesCO2')
                calif = elem.find('Calificacion/EnergiaPrimariaNoRenovable')
                
                # --- Lógica de Orientaciones (Corregida con Este) ---
                acristalada = geom.find('PorcentajeSuperficieAcristalada') if geom is not None else None
                try:
                    def get_v(node, tag):
                        if node is None: return 0
                        res = node.findtext(tag)
                        return float(res) if res and res.strip() else 0

                    pct_norte = get_v(acristalada, 'N') + get_v(acristalada, 'NO') + get_v(acristalada, 'NE') 
                    pct_sur   = get_v(acristalada, 'S') + get_v(acristalada, 'SO') + get_v(acristalada, 'SE')
                    pct_este  = get_v(acristalada, 'E')
                    pct_oeste = get_v(acristalada, 'O')
                except:
                    pct_norte, pct_sur, pct_este, pct_oeste = 0, 0, 0, 0

                # --- Extracción de datos de Instalaciones (Segura) ---
                # Buscamos el primer generador disponible
                gen = inst.find('.//GeneradoresDeCalefaccion/Generador') if inst is not None else None
                
                # Buscamos autoconsumo
                autocons = renov.find('.//Electrica/Sistema/EnergiaGeneradaAutoconsumida') if renov is not None else None

                # --- Construcción de la fila ---
                fila = {
                    'municipio': ident.findtext('Municipio') if ident is not None else None,
                    'provincia': ident.findtext('Provincia') if ident is not None else None,
                    'zona_climatica': ident.findtext('ZonaClimatica') if ident is not None else None,
                    'ano_construccion': ident.findtext('AnoConstruccion') if ident is not None else None,
                    'normativa': ident.findtext('NormativaVigente') if ident is not None else None,
                    'tipo_edificio': ident.findtext('TipoDeEdificio') if ident is not None else None,
                    
                    'superficie': geom.findtext('SuperficieHabitable') if geom is not None else None,
                    'compacidad': geom.findtext('Compacidad') if geom is not None else None,
                    'pct_calefactado': geom.findtext('PorcentajeSuperficieHabitableCalefactada') if geom is not None else None,
                    'pct_refrigerado': geom.findtext('PorcentajeSuperficieHabitableRefrigerada') if geom is not None else None,
                    
                    'ventana_norte': pct_norte,
                    'ventana_sur': pct_sur,
                    'ventana_este': pct_este,
                    'ventana_oeste': pct_oeste,
                    
                    'tipo_generador_cal': gen.findtext('Tipo') if gen is not None else None,
                    'rendimiento_cal': gen.findtext('RendimientoEstacional') if gen is not None else None,
                    'vector_energetico_cal': gen.findtext('VectorEnergetico') if gen is not None else None,
                    
                    'autoconsumo_elec': autocons.text if autocons is not None else 0,

                    'consumo_global': consumo.findtext('Global') if consumo is not None else None,
                    'demanda_calefaccion': demanda.findtext('Calefaccion') if demanda is not None else None,
                    'demanda_refrigeracion': demanda.findtext('Refrigeracion') if demanda is not None else None,
                    'emisiones_global': emisiones.findtext('Global') if emisiones is not None else None,
                    'clase_consumo': calif.findtext('Global') if calif is not None else None
                }

                writer.writerow(fila)
                count += 1
                
                # Limpieza de memoria (Crucial)
                elem.clear()
                if count % 20000 == 0:
                    print(f"✅ {count} registros procesados...")

    print(f"\n✨ ¡HECHO! Total: {count} registros guardados en {CSV_SALIDA}")


In [23]:

PROVINCIA = "ALMERIA"
XML_ENTRADA = f'../data/raw/{PROVINCIA}.xml'
CSV_SALIDA = f'../data/raw/cert_energ_{PROVINCIA.lower()}_rev2.csv'

# Definición de columnas finales
COLUMNAS = [
    'municipio', 'provincia', 'zona_climatica', 'ano_construccion', 'normativa', 'tipo_edificio',
    'superficie', 'compacidad', 'pct_calefactado', 'pct_refrigerado', 
    'ventana_norte', 'ventana_sur', 'ventana_este', 'ventana_oeste',
    'tipo_generador_cal', 'rendimiento_cal', 'vector_energetico_cal',
    'autoconsumo_elec', 'consumo_global', 'demanda_calefaccion', 
    'demanda_refrigeracion', 'emisiones_global', 'clase_consumo'
]

ejecutar_extraccion()

🚀 Iniciando extracción de ALMERIA...
📂 Origen: ../data/raw/ALMERIA.xml
📄 Destino: ../data/raw/cert_energ_almeria_rev2.csv
✅ 20000 registros procesados...
✅ 40000 registros procesados...
✅ 60000 registros procesados...
✅ 80000 registros procesados...
✅ 100000 registros procesados...

✨ ¡HECHO! Total: 115613 registros guardados en ../data/raw/cert_energ_almeria_rev2.csv


In [24]:
PROVINCIA = "CADIZ"
XML_ENTRADA = f'../data/raw/{PROVINCIA}.xml'
CSV_SALIDA = f'../data/raw/cert_energ_{PROVINCIA.lower()}_rev2.csv'

# Definición de columnas finales
COLUMNAS = [
    'municipio', 'provincia', 'zona_climatica', 'ano_construccion', 'normativa', 'tipo_edificio',
    'superficie', 'compacidad', 'pct_calefactado', 'pct_refrigerado', 
    'ventana_norte', 'ventana_sur', 'ventana_este', 'ventana_oeste',
    'tipo_generador_cal', 'rendimiento_cal', 'vector_energetico_cal',
    'autoconsumo_elec', 'consumo_global', 'demanda_calefaccion', 
    'demanda_refrigeracion', 'emisiones_global', 'clase_consumo'
]

ejecutar_extraccion()

🚀 Iniciando extracción de CADIZ...
📂 Origen: ../data/raw/CADIZ.xml
📄 Destino: ../data/raw/cert_energ_cadiz_rev2.csv
✅ 20000 registros procesados...
✅ 40000 registros procesados...
✅ 60000 registros procesados...
✅ 80000 registros procesados...
✅ 100000 registros procesados...
✅ 120000 registros procesados...
✅ 140000 registros procesados...

✨ ¡HECHO! Total: 154440 registros guardados en ../data/raw/cert_energ_cadiz_rev2.csv


In [25]:
PROVINCIA = "CORDOBA"
XML_ENTRADA = f'../data/raw/{PROVINCIA}.xml'
CSV_SALIDA = f'../data/raw/cert_energ_{PROVINCIA.lower()}_rev2.csv'

# Definición de columnas finales
COLUMNAS = [
    'municipio', 'provincia', 'zona_climatica', 'ano_construccion', 'normativa', 'tipo_edificio',
    'superficie', 'compacidad', 'pct_calefactado', 'pct_refrigerado', 
    'ventana_norte', 'ventana_sur', 'ventana_este', 'ventana_oeste',
    'tipo_generador_cal', 'rendimiento_cal', 'vector_energetico_cal',
    'autoconsumo_elec', 'consumo_global', 'demanda_calefaccion', 
    'demanda_refrigeracion', 'emisiones_global', 'clase_consumo'
]

ejecutar_extraccion()

🚀 Iniciando extracción de CORDOBA...
📂 Origen: ../data/raw/CORDOBA.xml
📄 Destino: ../data/raw/cert_energ_cordoba_rev2.csv
✅ 20000 registros procesados...
✅ 40000 registros procesados...
✅ 60000 registros procesados...

✨ ¡HECHO! Total: 74939 registros guardados en ../data/raw/cert_energ_cordoba_rev2.csv


In [26]:
PROVINCIA = "GRANADA"
XML_ENTRADA = f'../data/raw/{PROVINCIA}.xml'
CSV_SALIDA = f'../data/raw/cert_energ_{PROVINCIA.lower()}_rev2.csv'

# Definición de columnas finales
COLUMNAS = [
    'municipio', 'provincia', 'zona_climatica', 'ano_construccion', 'normativa', 'tipo_edificio',
    'superficie', 'compacidad', 'pct_calefactado', 'pct_refrigerado', 
    'ventana_norte', 'ventana_sur', 'ventana_este', 'ventana_oeste',
    'tipo_generador_cal', 'rendimiento_cal', 'vector_energetico_cal',
    'autoconsumo_elec', 'consumo_global', 'demanda_calefaccion', 
    'demanda_refrigeracion', 'emisiones_global', 'clase_consumo'
]

ejecutar_extraccion()

🚀 Iniciando extracción de GRANADA...
📂 Origen: ../data/raw/GRANADA.xml
📄 Destino: ../data/raw/cert_energ_granada_rev2.csv
✅ 20000 registros procesados...
✅ 40000 registros procesados...
✅ 60000 registros procesados...
✅ 80000 registros procesados...
✅ 100000 registros procesados...
✅ 120000 registros procesados...
✅ 140000 registros procesados...

✨ ¡HECHO! Total: 151706 registros guardados en ../data/raw/cert_energ_granada_rev2.csv


In [27]:
PROVINCIA = "HUELVA"
XML_ENTRADA = f'../data/raw/{PROVINCIA}.xml'
CSV_SALIDA = f'../data/raw/cert_energ_{PROVINCIA.lower()}_rev2.csv'

# Definición de columnas finales
COLUMNAS = [
    'municipio', 'provincia', 'zona_climatica', 'ano_construccion', 'normativa', 'tipo_edificio',
    'superficie', 'compacidad', 'pct_calefactado', 'pct_refrigerado', 
    'ventana_norte', 'ventana_sur', 'ventana_este', 'ventana_oeste',
    'tipo_generador_cal', 'rendimiento_cal', 'vector_energetico_cal',
    'autoconsumo_elec', 'consumo_global', 'demanda_calefaccion', 
    'demanda_refrigeracion', 'emisiones_global', 'clase_consumo'
]

ejecutar_extraccion()

🚀 Iniciando extracción de HUELVA...
📂 Origen: ../data/raw/HUELVA.xml
📄 Destino: ../data/raw/cert_energ_huelva_rev2.csv
✅ 20000 registros procesados...
✅ 40000 registros procesados...
✅ 60000 registros procesados...

✨ ¡HECHO! Total: 61951 registros guardados en ../data/raw/cert_energ_huelva_rev2.csv


In [28]:
PROVINCIA = "JAEN"
XML_ENTRADA = f'../data/raw/{PROVINCIA}.xml'
CSV_SALIDA = f'../data/raw/cert_energ_{PROVINCIA.lower()}_rev2.csv'

# Definición de columnas finales
COLUMNAS = [
    'municipio', 'provincia', 'zona_climatica', 'ano_construccion', 'normativa', 'tipo_edificio',
    'superficie', 'compacidad', 'pct_calefactado', 'pct_refrigerado', 
    'ventana_norte', 'ventana_sur', 'ventana_este', 'ventana_oeste',
    'tipo_generador_cal', 'rendimiento_cal', 'vector_energetico_cal',
    'autoconsumo_elec', 'consumo_global', 'demanda_calefaccion', 
    'demanda_refrigeracion', 'emisiones_global', 'clase_consumo'
]

ejecutar_extraccion()

🚀 Iniciando extracción de JAEN...
📂 Origen: ../data/raw/JAEN.xml
📄 Destino: ../data/raw/cert_energ_jaen_rev2.csv
✅ 20000 registros procesados...
✅ 40000 registros procesados...
✅ 60000 registros procesados...

✨ ¡HECHO! Total: 66590 registros guardados en ../data/raw/cert_energ_jaen_rev2.csv


In [29]:
PROVINCIA = "MALAGA"
XML_ENTRADA = f'../data/raw/{PROVINCIA}.xml'
CSV_SALIDA = f'../data/raw/cert_energ_{PROVINCIA.lower()}_rev2.csv'

# Definición de columnas finales
COLUMNAS = [
    'municipio', 'provincia', 'zona_climatica', 'ano_construccion', 'normativa', 'tipo_edificio',
    'superficie', 'compacidad', 'pct_calefactado', 'pct_refrigerado', 
    'ventana_norte', 'ventana_sur', 'ventana_este', 'ventana_oeste',
    'tipo_generador_cal', 'rendimiento_cal', 'vector_energetico_cal',
    'autoconsumo_elec', 'consumo_global', 'demanda_calefaccion', 
    'demanda_refrigeracion', 'emisiones_global', 'clase_consumo'
]

ejecutar_extraccion()

🚀 Iniciando extracción de MALAGA...
📂 Origen: ../data/raw/MALAGA.xml
📄 Destino: ../data/raw/cert_energ_malaga_rev2.csv
✅ 20000 registros procesados...
✅ 40000 registros procesados...
✅ 60000 registros procesados...
✅ 80000 registros procesados...
✅ 100000 registros procesados...
✅ 120000 registros procesados...
✅ 140000 registros procesados...
✅ 160000 registros procesados...
✅ 180000 registros procesados...
✅ 200000 registros procesados...
✅ 220000 registros procesados...
✅ 240000 registros procesados...
✅ 260000 registros procesados...
✅ 280000 registros procesados...
✅ 300000 registros procesados...

✨ ¡HECHO! Total: 309048 registros guardados en ../data/raw/cert_energ_malaga_rev2.csv


In [30]:
PROVINCIA = "SEVILLA"
XML_ENTRADA = f'../data/raw/{PROVINCIA}.xml'
CSV_SALIDA = f'../data/raw/cert_energ_{PROVINCIA.lower()}_rev2.csv'

# Definición de columnas finales
COLUMNAS = [
    'municipio', 'provincia', 'zona_climatica', 'ano_construccion', 'normativa', 'tipo_edificio',
    'superficie', 'compacidad', 'pct_calefactado', 'pct_refrigerado', 
    'ventana_norte', 'ventana_sur', 'ventana_este', 'ventana_oeste',
    'tipo_generador_cal', 'rendimiento_cal', 'vector_energetico_cal',
    'autoconsumo_elec', 'consumo_global', 'demanda_calefaccion', 
    'demanda_refrigeracion', 'emisiones_global', 'clase_consumo'
]

ejecutar_extraccion()

🚀 Iniciando extracción de SEVILLA...
📂 Origen: ../data/raw/SEVILLA.xml
📄 Destino: ../data/raw/cert_energ_sevilla_rev2.csv
✅ 20000 registros procesados...
✅ 40000 registros procesados...
✅ 60000 registros procesados...
✅ 80000 registros procesados...
✅ 100000 registros procesados...
✅ 120000 registros procesados...
✅ 140000 registros procesados...
✅ 160000 registros procesados...
✅ 180000 registros procesados...
✅ 200000 registros procesados...
✅ 220000 registros procesados...
✅ 240000 registros procesados...
✅ 260000 registros procesados...

✨ ¡HECHO! Total: 264735 registros guardados en ../data/raw/cert_energ_sevilla_rev2.csv


In [32]:
df_almeria = pd.read_csv('../data/raw/cert_energ_almeria_rev2.csv')
df_cadiz = pd.read_csv('../data/raw/cert_energ_cadiz_rev2.csv')
df_cordoba = pd.read_csv('../data/raw/cert_energ_cordoba_rev2.csv')
df_granada = pd.read_csv('../data/raw/cert_energ_granada_rev2.csv')
df_huelva = pd.read_csv('../data/raw/cert_energ_huelva_rev2.csv')
df_malaga = pd.read_csv('../data/raw/cert_energ_malaga_rev2.csv')
df_jaen = pd.read_csv('../data/raw/cert_energ_jaen_rev2.csv') 
df_sevilla = pd.read_csv('../data/raw/cert_energ_sevilla_rev2.csv')

In [34]:
print(f'📊 Dimensiones por provincia:')
print(f'   - Almería: {df_almeria.shape[0]} registros, {df_almeria.shape[1]} columnas')
print(f'   - Cádiz: {df_cadiz.shape[0]} registros, {df_cadiz.shape[1]} columnas')
print(f'   - Córdoba: {df_cordoba.shape[0]} registros, {df_cordoba.shape[1]} columnas')
print(f'   - Granada: {df_granada.shape[0]} registros, {df_granada.shape[1]} columnas')
print(f'   - Huelva: {df_huelva.shape[0]} registros, {df_huelva.shape[1]} columnas')
print(f'   - Málaga: {df_malaga.shape[0]} registros, {df_malaga.shape[1]} columnas')
print(f'   - Jaén: {df_jaen.shape[0]} registros, {df_jaen.shape[1]} columnas')
print(f'   - Sevilla: {df_sevilla.shape[0]} registros, {df_sevilla.shape[1]} columnas')

📊 Dimensiones por provincia:
   - Almería: 115613 registros, 23 columnas
   - Cádiz: 154440 registros, 23 columnas
   - Córdoba: 74939 registros, 23 columnas
   - Granada: 151706 registros, 23 columnas
   - Huelva: 61951 registros, 23 columnas
   - Málaga: 309048 registros, 23 columnas
   - Jaén: 66590 registros, 23 columnas
   - Sevilla: 264735 registros, 23 columnas


In [35]:
df_andalucia = pd.concat([df_almeria, df_cadiz, df_cordoba, df_granada, df_huelva, df_malaga, df_jaen, df_sevilla], ignore_index=True)
print(f'\n📊 Dimensión total de Andalucía: {df_andalucia.shape[0]} registros')


📊 Dimensión total de Andalucía: 1199022 registros


In [37]:

cols_numericas = [
    'superficie', 'compacidad', 'pct_calefactado', 'pct_refrigerado', 
    'ventana_norte', 'ventana_sur', 'ventana_este', 'ventana_oeste',
    'rendimiento_cal', 'vector_energetico_cal',
    'autoconsumo_elec', 'consumo_global', 'demanda_calefaccion', 
    'demanda_refrigeracion', 'emisiones_global'
]

for col in cols_numericas:
    df_andalucia[col] = pd.to_numeric(df_andalucia[col], errors='coerce')

df_andalucia.to_csv('../data/raw/cert_energ_andalucia.csv', index=False)